# Fine-tune PP-OCRv5_server_det on Kaggle

Notebook nay duoc thiet ke cho cach dung `Run All` tren Kaggle.

Muc tieu:

- fine-tune `PP-OCRv5_server_det` tren Kaggle
- lay checkpoint train ve local
- evaluate/export/infer local se lam sau

Cac file can tai ve sau khi train:

- `output/<run_name>/best_accuracy/best_accuracy.pdparams`
- `output/<run_name>/config.yaml`


### Cach dung

1. Gan Kaggle dataset chua `images/`, `train.txt`, `val.txt`
2. Chi sua cac bien trong cell `Configuration` ben duoi
3. Bam `Run All`


In [ ]:
from pathlib import Path

# ------------------------------
# Configuration
# ------------------------------
ROOT_DIR = Path("/kaggle/working/PaddleX")
DATASET_DIR = Path("/kaggle/input/my-ocr-det-dataset")
PRETRAIN_INPUT_PATH = Path("/kaggle/input/my-pretrained/PP-OCRv5_server_det_pretrained.pdparams")
PRETRAIN_URL = "https://paddle-model-ecology.bj.bcebos.com/paddlex/official_pretrained_model/PP-OCRv5_server_det_pretrained.pdparams"
PRETRAIN_WORK_PATH = Path("/kaggle/working/pretrained/PP-OCRv5_server_det_pretrained.pdparams")
OUTPUT_DIR = Path("/kaggle/working/output/ppocrv5_server_det")

DEVICE = "gpu:0"
EPOCHS = 100
BATCH_SIZE = 4
LEARNING_RATE = 0.001
EVAL_INTERVAL = 1
SAVE_INTERVAL = 1
LOG_INTERVAL = 10

print("ROOT_DIR           =", ROOT_DIR)
print("DATASET_DIR        =", DATASET_DIR)
print("PRETRAIN_INPUT     =", PRETRAIN_INPUT_PATH)
print("PRETRAIN_WORK_PATH =", PRETRAIN_WORK_PATH)
print("OUTPUT_DIR         =", OUTPUT_DIR)
print("DEVICE             =", DEVICE)
print("EPOCHS             =", EPOCHS)
print("BATCH_SIZE         =", BATCH_SIZE)
print("LEARNING_RATE      =", LEARNING_RATE)


## 1. Validate Inputs

Cell nay fail som neu dataset chua duoc mount dung vao Kaggle input.

Voi pretrained weight, notebook ho tro 2 truong hop:

- da co san trong Kaggle Input
- chua co thi tu tai ve `/kaggle/working/pretrained/`


In [ ]:
import urllib.request

required_dataset_paths = [
    (DATASET_DIR, "Dataset directory"),
    (DATASET_DIR / "train.txt", "train.txt"),
    (DATASET_DIR / "val.txt", "val.txt"),
    (DATASET_DIR / "images", "images directory"),
]

missing = []
for path, desc in required_dataset_paths:
    if not path.exists():
        missing.append(f"{desc} not found: {path}")

if missing:
    raise FileNotFoundError("\n".join(missing))

if PRETRAIN_INPUT_PATH.exists():
    FINAL_PRETRAIN_PATH = PRETRAIN_INPUT_PATH
    print("Using pretrained weight from Kaggle Input:", FINAL_PRETRAIN_PATH)
else:
    PRETRAIN_WORK_PATH.parent.mkdir(parents=True, exist_ok=True)
    if not PRETRAIN_WORK_PATH.exists():
        print("Pretrained weight not found in Input. Downloading from official URL...")
        urllib.request.urlretrieve(PRETRAIN_URL, PRETRAIN_WORK_PATH)
    FINAL_PRETRAIN_PATH = PRETRAIN_WORK_PATH
    print("Using downloaded pretrained weight:", FINAL_PRETRAIN_PATH)

print("Train annotation:", DATASET_DIR / "train.txt")
print("Val annotation  :", DATASET_DIR / "val.txt")
print("Images dir      :", DATASET_DIR / "images")
print("Final pretrained:", FINAL_PRETRAIN_PATH)


## 2. Clone PaddleX


In [ ]:
%%bash
set -euo pipefail
cd /kaggle/working

if [ ! -d /kaggle/working/PaddleX ]; then
  git clone https://github.com/PaddlePaddle/PaddleX.git
fi

cd /kaggle/working/PaddleX
git rev-parse HEAD


## 3. Install Environment

Stack nay bam theo luong PaddleX text detection tren Kaggle:

- `paddlepaddle-gpu==3.3.0`
- `pip install -e \".[base]\"`
- `paddlex --install PaddleOCR -y`
- `setuptools<81`


In [ ]:
%%bash
set -euo pipefail
cd /kaggle/working/PaddleX

python -m pip install --upgrade pip setuptools wheel
python -m pip install paddlepaddle-gpu==3.3.0 -i https://www.paddlepaddle.org.cn/packages/stable/cu118/
python -m pip install -e ".[base]"
paddlex --install PaddleOCR -y
python -m pip install "setuptools<81"


## 4. Check Dataset with PaddleX

Buoc nay phat hien loi format `train.txt` / `val.txt` truoc khi train.


In [ ]:
import os
import subprocess

env = os.environ.copy()
env.update({
    "HOME": "/kaggle/working/.home",
    "PADDLE_PDX_CACHE_HOME": "/kaggle/working/.paddlex_cache",
    "MPLCONFIGDIR": "/kaggle/working/.matplotlib",
})

for d in [env["HOME"], env["PADDLE_PDX_CACHE_HOME"], env["MPLCONFIGDIR"], str(OUTPUT_DIR)]:
    Path(d).mkdir(parents=True, exist_ok=True)

cmd = [
    "python",
    str(ROOT_DIR / "main.py"),
    "-c", str(ROOT_DIR / "paddlex/configs/modules/text_detection/PP-OCRv5_server_det.yaml"),
    "-o", "Global.mode=check_dataset",
    "-o", f"Global.device={DEVICE}",
    "-o", f"Global.dataset_dir={DATASET_DIR}",
]

print("Running:")
print(" ".join(map(str, cmd)))
subprocess.run(cmd, env=env, cwd=ROOT_DIR, check=True)


## 5. Train

Cell nay bat dau fine-tune.

Muc tieu output sau khi chay xong:

- `OUTPUT_DIR / config.yaml`
- `OUTPUT_DIR / best_accuracy / best_accuracy.pdparams`


In [ ]:
train_cmd = [
    "python",
    str(ROOT_DIR / "main.py"),
    "-c", str(ROOT_DIR / "paddlex/configs/modules/text_detection/PP-OCRv5_server_det.yaml"),
    "-o", "Global.mode=train",
    "-o", f"Global.device={DEVICE}",
    "-o", f"Global.dataset_dir={DATASET_DIR}",
    "-o", f"Global.output={OUTPUT_DIR}",
    "-o", f"Train.epochs_iters={EPOCHS}",
    "-o", f"Train.batch_size={BATCH_SIZE}",
    "-o", f"Train.learning_rate={LEARNING_RATE}",
    "-o", f"Train.eval_interval={EVAL_INTERVAL}",
    "-o", f"Train.save_interval={SAVE_INTERVAL}",
    "-o", f"Train.log_interval={LOG_INTERVAL}",
    "-o", f"Train.pretrain_weight_path={FINAL_PRETRAIN_PATH}",
]

print("Running:")
print(" ".join(map(str, train_cmd)))
subprocess.run(train_cmd, env=env, cwd=ROOT_DIR, check=True)


## 6. Verify Outputs

Buoc nay xac nhan co du file can tai ve local.


In [ ]:
expected_files = [
    OUTPUT_DIR / "config.yaml",
    OUTPUT_DIR / "best_accuracy" / "best_accuracy.pdparams",
]

missing_outputs = [str(p) for p in expected_files if not p.exists()]
if missing_outputs:
    raise FileNotFoundError("Missing expected output files:\n" + "\n".join(missing_outputs))

print("Expected output files found.")
for p in expected_files:
    print("-", p)


In [ ]:
%%bash
set -euo pipefail
echo "== output tree =="
find /kaggle/working/output -maxdepth 4 | sort


## 7. Zip Output For Download

Khuyen nghi tai ca thu muc output cua run train, khong chi rieng file weight.


In [ ]:
import zipfile

zip_path = Path("/kaggle/working/ppocrv5_server_det_output.zip")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for path in OUTPUT_DIR.rglob("*"):
        if path.is_file():
            zf.write(path, path.relative_to(OUTPUT_DIR.parent))

print("Saved zip to:", zip_path)


## 8. What To Download Back To Local

Toi thieu:

```text
output/<run_name>/best_accuracy/best_accuracy.pdparams
output/<run_name>/config.yaml
```

Khuyen nghi:

- tai ca file zip vua tao
- neu da co `best_accuracy/inference` thi cung nen giu lai de infer nhanh
- evaluate/export local se lam sau de tranh lech env
